<a href="https://colab.research.google.com/github/kader-xai/data-science-roadmap/blob/main/module_84_rnn_lstm_gru_deep_dive.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> 🌀 **Module 84 — RNN / LSTM / GRU Deep Dive** &nbsp;·&nbsp; part of [`data-science-roadmap`](https://github.com/kader-xai/data-science-roadmap)

> Phase 11 · Foundations Backfill


# Module 84 — RNN / LSTM / GRU Deep Dive

> M22 trained an LSTM forecaster as one part of the time-series module, but no module ever explained **why gates work** or **why we needed LSTMs at all**. This module is that depth pass. Twenty years of sequence-model history compressed into one notebook — and the answer to "in 2026, when do RNNs still beat Transformers?" (the answer is **yes, sometimes**).

### What you'll cover
1. Why sequence models — and the inductive bias they bring
2. The **vanilla RNN cell** — runnable in 8 lines
3. **Backprop through time (BPTT)** + the **vanishing / exploding gradient** problem
4. **LSTM** — gates and why they work
5. **GRU** — the simpler, often-equal alternative
6. **Bidirectional** + **stacked** + **dropout** RNNs
7. **Seq2seq with attention** (Bahdanau / Luong) — the 2014–2015 era that pre-figured Transformers
8. The **`nn.LSTM`** API (and the four shape gotchas)
9. Where RNNs **still beat Transformers in 2026** — six real cases
10. The **modern recurrence revival** — Mamba / SSMs / xLSTM / RWKV / RetNet


## 1 · Why sequence models

A fully-connected net treats `[x₀, x₁, …, x_T]` as one big vector — no notion of order. A CNN (M83) brings translation equivariance for spatial data, but a **shifted timeseries is not the same signal** in the way a shifted image is.

What sequence models bring:
- **Variable-length input** — you can run them on a 5-step sequence or a 5 000-step one.
- **Causality (left-to-right)** — at time `t`, the model has only seen `x₀..x_t`.
- **State carried forward** — the model summarises everything it's seen so far into a fixed-size **hidden state** `h_t`.

That last property is the win. Compression of the past into a hidden state is what makes RNN inference **O(1) per token** — vs Transformer self-attention's **O(N)**. We'll see why this matters in §9.

## 2 · The vanilla RNN cell

The simplest possible recurrent cell:

$$h_t = \tanh(W_x x_t + W_h h_{t-1} + b)$$

One linear combination, one nonlinearity, output is the new state. That's it.

In [ ]:
import torch, torch.nn as nn

class VanillaRNNCell(nn.Module):
    def __init__(self, x_dim, h_dim):
        super().__init__()
        self.Wx = nn.Linear(x_dim, h_dim, bias=False)
        self.Wh = nn.Linear(h_dim, h_dim, bias=True)
    def forward(self, x_t, h_prev):
        return torch.tanh(self.Wx(x_t) + self.Wh(h_prev))

# Roll over a sequence of length T
def run_rnn(cell, x_seq):
    B, T, _ = x_seq.shape
    h = torch.zeros(B, cell.Wh.out_features)
    outputs = []
    for t in range(T):
        h = cell(x_seq[:, t], h)
        outputs.append(h)
    return torch.stack(outputs, dim=1), h     # (B, T, H), (B, H)

cell = VanillaRNNCell(x_dim=8, h_dim=16)
x = torch.randn(4, 20, 8)                       # batch 4, length 20, features 8
out, h_final = run_rnn(cell, x)
print(out.shape, h_final.shape)

Three things to notice:

- The cell uses the **same `Wx` and `Wh`** at every timestep — that's the analogue of CNN weight sharing (M83).
- The hidden-state size is your **memory capacity** — bigger is better up to a point, then overfits.
- There's no nonlinearity *between* timesteps in any direction except `tanh`. That matters in §3.

## 3 · Backprop through time + the vanishing / exploding gradient problem

To train, we unroll the cell over `T` steps and call `loss.backward()`. The gradient at step 0 must flow back through every multiplication by `W_h` (and the `tanh` derivative). After `T` steps:

$$\frac{\partial h_T}{\partial h_0} = \prod_{t=1}^{T} W_h^\top \cdot \text{diag}(1 - h_t^2)$$

Two failure modes:

| Failure | When | Symptom |
|---|---|---|
| **Vanishing gradient** | spectral radius of the product < 1 | step-0 gradient ≈ 0; the model can't learn long-range dependencies |
| **Exploding gradient** | spectral radius > 1 | gradient → ∞; loss = NaN |

**Mitigations:**
- **Gradient clipping** (`torch.nn.utils.clip_grad_norm_`) — caps the magnitude. Cheap. Always-on.
- **Truncated BPTT** — backprop only through the last `k` steps. Cuts compute + bounds the explosion.
- **Better activations / inits** (`tanh + orthogonal init`) — marginal.
- **The structural fix: LSTM / GRU.** ← what the next two sections build.

In [ ]:
# Always pair RNN training with gradient clipping:
for x, y in loader:
    opt.zero_grad()
    pred = model(x)
    loss = criterion(pred, y)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)   # ← the line
    opt.step()

## 4 · LSTM — Long Short-Term Memory

Hochreiter & Schmidhuber (1997) — a structural fix for vanishing gradients. The LSTM adds a **cell state** `c_t` that flows along the top with **only additive updates**, gated by sigmoid functions:

```
                    ┌──────────────────────────────────────────────┐
                    │                                              │
   x_t          ───►│   ┌──── forget ────► × ──────────────────┐  │
                    │   │                                       │  │
   h_{t-1}      ───►│   ├──── input  ────► × ──── tanh ───┐    │  │
                    │   │                                  │    ▼  │
                    │   │                                  ▼    +  │── c_t  ───►
                    │   │                                  ⊕────┴  │
                    │   │                                       │  │
                    │   └──── output  ────► × ◄──── tanh ◄──────┤  │── h_t  ───►
                    │                                            │  │
                    └────────────────────────────────────────────┘
```

The three **gates** (sigmoid → values in [0, 1]):

- **Forget gate** `f_t` — what to drop from the cell state.
- **Input gate** `i_t` — what new candidate `c̃_t` to write.
- **Output gate** `o_t` — what to expose to the rest of the network as `h_t`.

Equations:

$$\begin{aligned}
f_t &= \sigma(W_f \cdot [h_{t-1}, x_t] + b_f) \\
i_t &= \sigma(W_i \cdot [h_{t-1}, x_t] + b_i) \\
\tilde{c}_t &= \tanh(W_c \cdot [h_{t-1}, x_t] + b_c) \\
c_t &= f_t \odot c_{t-1} + i_t \odot \tilde{c}_t \\
o_t &= \sigma(W_o \cdot [h_{t-1}, x_t] + b_o) \\
h_t &= o_t \odot \tanh(c_t)
\end{aligned}$$

**Why this fixes vanishing gradients.** The cell state `c_t` is updated *additively* (the `+` in the diagram), gated by `f_t`. When `f_t ≈ 1`, the gradient flows through `c` essentially unchanged — gradients can travel hundreds of steps without decay. That's the whole point.

In [ ]:
class LSTMCell(nn.Module):
    """From-scratch LSTM cell. PyTorch's nn.LSTMCell is the production version."""
    def __init__(self, x_dim, h_dim):
        super().__init__()
        # Combine all four gate projections into one matmul for speed
        self.W = nn.Linear(x_dim + h_dim, 4 * h_dim)
        self.h_dim = h_dim
    def forward(self, x_t, state):
        h_prev, c_prev = state
        gates = self.W(torch.cat([x_t, h_prev], dim=-1))
        i, f, g, o = gates.chunk(4, dim=-1)         # input, forget, candidate, output
        i, f, o = torch.sigmoid(i), torch.sigmoid(f), torch.sigmoid(o)
        g = torch.tanh(g)
        c = f * c_prev + i * g
        h = o * torch.tanh(c)
        return h, (h, c)

cell = LSTMCell(8, 16)
h, c = torch.zeros(4, 16), torch.zeros(4, 16)
for t in range(20):
    out, (h, c) = cell(torch.randn(4, 8), (h, c))
print("final hidden:", h.shape, "cell:", c.shape)

**A practical insight.** Most production LSTM code initialises the **forget-gate bias to ~1**:
```python
nn.init.constant_(lstm.bias_ih_l0[h_dim:2*h_dim], 1.0)
```
This makes the forget gate start near `σ(1) ≈ 0.73`, so the cell state mostly persists at init — the network can learn what to forget rather than starting at "forget everything." Small change, big training stability.

## 5 · GRU — Gated Recurrent Unit

Cho et al. (2014). Same idea as LSTM with **one fewer gate** and **no separate cell state**. Two gates:

- **Update gate** `z_t` — mixes the previous hidden state with the new candidate.
- **Reset gate** `r_t` — allows the cell to "forget" the previous hidden state when computing the candidate.

$$\begin{aligned}
z_t &= \sigma(W_z [h_{t-1}, x_t]) \\
r_t &= \sigma(W_r [h_{t-1}, x_t]) \\
\tilde{h}_t &= \tanh(W_h [r_t \odot h_{t-1}, x_t]) \\
h_t &= (1 - z_t) \odot h_{t-1} + z_t \odot \tilde{h}_t
\end{aligned}$$

| | LSTM | GRU |
|---|---|---|
| Gates | 3 (forget, input, output) | 2 (update, reset) |
| Has cell state? | yes (separate `c_t`) | no |
| Parameters | ~4·d² | ~3·d² (~25% less) |
| When wins | very long sequences, strong memory needed | shorter sequences, less data, faster training |

**The 2026 honest take.** On most tasks they tie. **GRU** is the better default if you're starting from scratch — smaller, simpler. **LSTM** is what every paper from 1997-2017 used, so the literature is bigger.

## 6 · Bidirectional + stacked + dropout

Three orthogonal extensions you'll meet in every production RNN:

### Bidirectional RNN
Two RNNs run in opposite directions; their outputs are concatenated. Doubles parameters; lets `h_t` see *future* tokens. Only valid for **non-causal** tasks (offline NER, sentiment, machine translation encoder). **Forbidden** for streaming or autoregressive generation.

```python
lstm = nn.LSTM(input_size=8, hidden_size=16, bidirectional=True)
```

### Stacked / deep RNNs
Output of layer `l` is the input of layer `l+1`. Increases expressive power, raises perplexity-vs-params efficiency.

```python
lstm = nn.LSTM(input_size=8, hidden_size=16, num_layers=4)
```

### Dropout — *recurrent* dropout
`nn.LSTM(..., dropout=p)` applies dropout **between layers**, not between timesteps. For dropout *between timesteps* (the right thing for regularising overfit), use **variational dropout** (Gal & Ghahramani) — same mask reused at every step.

## 7 · Seq2seq with attention — the bridge to Transformers

The 2014 **encoder-decoder** architecture (Sutskever et al.) was the first major sequence-to-sequence model. One LSTM **encodes** the source sentence into a single fixed-size vector; a second LSTM **decodes** the target sentence from it.

Two problems:
1. **Fixed-size bottleneck** — compressing a 50-word sentence into a single vector loses information.
2. **Long-range dependency** — decoder must remember the start of the source through all decoder steps.

**Bahdanau attention (2014)** — the fix that opened the modern era:

```
   decoder step t:
     attention weights α_{t,j} = softmax_j(score(h^dec_t, h^enc_j))
     context c_t = Σ_j α_{t,j} · h^enc_j         ← weighted sum of encoder states
     decoder uses [h^dec_t, c_t] to predict next token
```

The decoder doesn't read one summary vector — it reads **a different soft mixture of encoder states** at each step. Score functions: dot-product, additive (Bahdanau), multiplicative (Luong).

**Luong's 2015 simplification** (`score = h_t^T W h_j`) was directly adopted by the 2017 Transformer (M19, M20). When you read scaled-dot-product attention in M19, you're looking at Luong attention scaled by `√d_k` — applied to *every pair* (not just decoder ↔ encoder). The recurrence was the only piece that went away.

## 8 · `nn.LSTM` — the production API + four shape gotchas

In [ ]:
# Production LSTM in PyTorch
lstm = nn.LSTM(
    input_size=64,
    hidden_size=128,
    num_layers=2,
    dropout=0.2,
    bidirectional=False,
    batch_first=True,                   # ← (B, T, F) instead of (T, B, F)
)

# Shape gotcha 1: batch_first
x = torch.randn(32, 50, 64)             # (B=32, T=50, F=64)
y, (h_n, c_n) = lstm(x)

# Shape gotcha 2: h_n / c_n shape is (num_layers * num_directions, B, H), NOT batch-first
print("y shape:", y.shape)              # (B, T, H * num_directions)
print("h_n shape:", h_n.shape)          # (num_layers * num_directions, B, H)

# Shape gotcha 3: hidden state is per-LAYER, not per-timestep — usually you want y[:, -1]
last_hidden = y[:, -1, :]               # the canonical "summary vector" for classification
print("last hidden:", last_hidden.shape)

# Shape gotcha 4: PACKED sequences — variable length without padding waste
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
lengths = torch.tensor([50, 47, 42, 30, 15])    # actual lengths per batch row
# (caller must pre-sort the batch by descending length in older PyTorch)
packed = pack_padded_sequence(x[:5], lengths, batch_first=True, enforce_sorted=True)
packed_out, _ = lstm(packed)
y_unpacked, _ = pad_packed_sequence(packed_out, batch_first=True)
print("unpacked:", y_unpacked.shape)

**The four shape rules to keep on a sticky note:**
1. `batch_first=True` → input + output are `(B, T, F)`. Without it, defaults are `(T, B, F)`. **Set `batch_first=True`.**
2. `h_n` and `c_n` are always `(num_layers × directions, B, hidden_size)` — never batch-first regardless of the flag.
3. For classification, **use `y[:, -1, :]`** (the last timestep's output) — *not* `h_n` for stacked / bidirectional LSTMs (it gets confusing fast).
4. For ragged batches, **pack** with `pack_padded_sequence` — otherwise PyTorch wastes compute on padding.

## 9 · Where RNNs still beat Transformers in 2026

Transformers won most of NLP, but RNNs **never went away**. Six places they're still the right call:

| Use case | Why RNN wins |
|---|---|
| **Streaming / real-time inference** | RNN is **O(1) per token**; Transformer is **O(N)** without KV cache. For ASR (M65 Whisper streaming), real-time captioning, online anomaly detection, RNNs lead. |
| **Very long sequences with limited memory** | Transformer self-attention is **O(N²)** memory. An LSTM is **O(1)**. For long ECG / sensor / power-grid traces, RNNs (or SSMs) win. |
| **Tiny model / on-device** | A 2-layer LSTM with 100 K params runs on a microcontroller; the smallest Transformer doesn't. M90 (edge AI) cares. |
| **Time-series forecasting** | Many production systems still use LSTM-based DeepAR / Temporal Fusion Transformer (which uses LSTM as a sub-block) — they win on tabular-time-series benchmarks consistently. |
| **Limited data** | RNN's strong inductive bias for sequences trains with less data than a vanilla Transformer (though a *small* Transformer with appropriate norms can match it). |
| **Hierarchical / nested structures** | TreeLSTMs and recursive networks still appear in syntax and program analysis. |

**`Transformer ≠ always the answer.`** When asked "should I use a Transformer here?" you should be able to ask "what's the sequence length, the batch latency target, and the inference budget?" If long + tight + small, **RNN or SSM is often the call**.

## 10 · The modern recurrence revival — Mamba, SSMs, xLSTM, RWKV, RetNet

After 5 years of "Transformer is all you need," 2023–25 brought a **renaissance of recurrence** under new names:

| Family | Key idea | When it wins |
|---|---|---|
| **Linear Attention** (Katharopoulos 2020) | replace softmax-attention with a linear-kernel variant; gives O(N) memory & a recurrent form for inference | sequence modelling at length |
| **RetNet** (Microsoft, 2023) | retention mechanism; trainable in parallel like Transformer, recurrent at inference like RNN | LLM-style models with long-context inference |
| **RWKV** (open-source, 2022+) | linear attention reformulated as RNN; runs LLM-quality models with O(1)-per-token inference | community / edge LLMs |
| **Mamba / SSMs** (Gu & Dao, 2023+) | state-space models with **selective** state updates; matches Transformer at < 7B; better for long contexts | DNA, audio, long documents |
| **xLSTM** (Hochreiter 2024) | LSTM with exponential gating and matrix memory; competitive at LLM scale | LLM-style, long context |
| **Hybrid models** (Jamba, Zamba, Samba) | mix Transformer blocks with Mamba blocks — best of both | most 2025 production "long-context" models |

**The takeaway**: the *mechanism* of recurrence wasn't replaced by Transformers — only the **specific tanh+gates parametrisation** was. With the right state-space or linear-attention parametrisation, **a recurrent model can match a Transformer's quality with much better inference economics**. M20's KV-cache + M44's PagedAttention + M77's GPU networking exist *because* Transformer inference is expensive at long context; SSMs sidestep most of that.

## ✅ Recap

- Sequence models carry **causality**, **variable length**, and **compressed state** — that's the inductive bias.
- The vanilla RNN's `h_t = tanh(W_x x_t + W_h h_{t-1})` is elegant but suffers **vanishing / exploding gradients** during BPTT.
- **LSTM** fixes vanishing gradients with **additive cell state + gates**. **GRU** is a 25%-smaller equivalent.
- Always pair RNN training with **gradient clipping**. Initialise the **forget-gate bias near 1**.
- **Bidirectional**, **stacked**, and **variational dropout** are the three production extensions.
- **Bahdanau / Luong attention** on top of LSTM seq2seq was the direct ancestor of the Transformer.
- The **`nn.LSTM` API** has 4 shape gotchas: `batch_first`, the `h_n` shape, `y[:, -1]` for classification, `pack_padded_sequence` for ragged batches.
- RNNs still win in **streaming**, **very long sequences**, **on-device**, **time-series**, **limited-data**, and **hierarchical-structure** scenarios.
- The 2023+ **Mamba / SSM / xLSTM / RWKV / RetNet** revival shows recurrence is *back*, just under new parametrisations.

Next: **M85 — GANs · Autoencoders · VAE**.
